In [8]:
# I'm setting up my notebook with my correct Project ID.
import os
PROJECT_ID = "qwiklabs-gcp-01-ed33dcbea5fb"
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

# This special command loads the BigQuery 'magic' so I can run SQL in my notebook cells.
%load_ext google.cloud.bigquery

print(f"My environment is set up for project: {PROJECT_ID}")


/usr/local/lib/python3.12/dist-packages/google/cloud/bigquery/__init__.py:239: FutureWarning: %load_ext google.cloud.bigquery is deprecated. Install bigquery-magics package and use `%load_ext bigquery_magics`, instead.
  warnings.warn(


My environment is set up for project: qwiklabs-gcp-01-ed33dcbea5fb


In [9]:
%%bigquery
-- I'm creating a new dataset called 'emergency_response_ch2' to keep my work organized.

CREATE SCHEMA IF NOT EXISTS emergency_response_ch2
OPTIONS(location="US");


Query is running:   0%|          |

""


In [11]:
%%bigquery
-- I'm telling BigQuery to load the data directly from the lab's GCS bucket into my new table.
LOAD DATA OVERWRITE emergency_response_ch2.raw_emergency_calls
FROM FILES (
  format = 'CSV',
  uris = ['gs://labs.roitraining.com/data-to-ai-workshop/emergency_calls_response_times.csv'],
  skip_leading_rows = 1
);


Query is running:   0%|          |

""


In [12]:
%%bigquery
-- I'm selecting the first 10 rows from my table to see what the data looks like.
SELECT * FROM emergency_response_ch2.raw_emergency_calls LIMIT 10;


Query is running:   0%|          |

Downloading:   0%|          |

,call_id,call_timestamp,call_type,location,weather_condition,day_of_week,time_of_day,traffic_level,distance_to_station,units_available,response_time
0,35957,2023-01-01 00:05:53+00:00,Fire,Highland,Rainy,Sunday,0,High,21.45,3,23.41
1,20832,2023-01-01 00:20:47+00:00,Fire,Oakmont,Rainy,Sunday,0,High,22.29,6,20.11
2,27949,2023-01-01 00:33:27+00:00,Fire,Riverside,Windy,Sunday,0,High,17.19,14,19.75
3,20199,2023-01-01 00:48:29+00:00,Fire,Riverside,Windy,Sunday,0,High,17.39,14,20.76
4,46938,2023-01-01 00:50:44+00:00,Rescue,Brookfield,Sunny,Sunday,0,High,22.50,14,22.37
5,17582,2023-01-01 02:28:50+00:00,Rescue,Downtown,Snowy,Sunday,2,High,25.15,6,28.48
6,21624,2023-01-01 02:44:06+00:00,Rescue,Oakmont,Snowy,Sunday,2,High,3.95,9,19.30
7,36793,2023-01-01 02:53:54+00:00,Fire,Riverside,Sunny,Sunday,2,High,5.87,10,10.72
8,41350,2023-01-01 03:52:33+00:00,Police,Greenfield,Windy,Sunday,3,High,6.66,5,20.55
9,32092,2023-01-01 04:09:23+00:00,Police,Maplewood,Snowy,Sunday,4,High,15.50,13,22.98


In [15]:
%%bigquery
-- I'm creating a new model to predict the 'response_time'.
CREATE OR REPLACE MODEL emergency_response_ch2.response_time_prediction_model
OPTIONS(
  model_type='BOOSTED_TREE_REGRESSOR',
  input_label_cols=['response_time']
) AS
SELECT
  -- These are the features my model will use to make its predictions.
  call_type,
  day_of_week,

  CAST(time_of_day AS STRING) AS time_of_day_str,
  traffic_level,
  distance_to_station,
  units_available,
  weather_condition,
  response_time -- I also need to include my target column in the SELECT statement.
FROM
  emergency_response_ch2.raw_emergency_calls
WHERE
  response_time IS NOT NULL; -- It's critical to only train my model on data where I actually know the answer.


Query is running:   0%|          |

""


In [16]:
%%bigquery
SELECT * FROM ML.EVALUATE(MODEL emergency_response_ch2.response_time_prediction_model);


Query is running:   0%|          |

Downloading:   0%|          |

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,1.892299,5.643337,0.018208,1.578528,0.798722,0.799837
